# RIDI (Reproducibility of Identity Decisions Index) in 60 seconds

**Question:** Can two rankings have near-perfect global agreement while allocating every finite decision seat to different identities?

This notebook constructs the counterexample directly. It is deterministic, uses no external data, and makes no model assumptions.

In [ ]:
# Change these two values and rerun. The only constraint is 2*k <= n.
n = 10_000   # ranked candidate universe
k = 50       # finite decision capacity
assert 1 <= k <= n // 2

In [ ]:
# R0 is the baseline ranking. R1 swaps only the first two adjacent blocks of size k.
r0 = list(range(n))
r1 = r0[k:2*k] + r0[:k] + r0[2*k:]

A, B = set(r0[:k]), set(r1[:k])
overlap = len(A & B)
rho = 1 - 12*k**3 / (n*(n**2 - 1))
ridi = 1 - overlap / len(A | B)

print(f'Candidates (n):              {n:,}')
print(f'Decision capacity (k):       {k:,}')
print(f'Global Spearman agreement:   {rho:.9f}')
print(f'Top-k overlap:               {overlap}/{k}')
print(f'RIDI:                        {ridi:.3f}')

## What happened?

Only `2k` candidates moved, and each moved exactly `k` ranks. The remaining `n-2k` candidates did not move at all. That small global disturbance is enough to replace the entire top-*k*.

For this construction:

$$\rho = 1 - \frac{12k^3}{n(n^2-1)} \rightarrow 1 \quad \text{as } n \rightarrow \infty,$$

while $\mathrm{RIDI}=1$ for every valid `n` and `k`. Global rank agreement therefore cannot certify decision identity.

In [ ]:
# See the divergence across candidate-universe sizes.
import matplotlib.pyplot as plt

universes = [100, 250, 500, 1_000, 2_500, 5_000, 10_000, 50_000]
rhos = [1 - 12*k**3 / (size*(size**2 - 1)) for size in universes]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(universes, rhos, marker='o', linewidth=2.5, color='#1769aa')
ax.axhline(1, color='#d96500', linestyle='--', linewidth=1.5, label='Perfect global agreement')
ax.set_xscale('log')
ax.set_xlabel('Candidate universe n (log scale)')
ax.set_ylabel('Spearman agreement')
ax.set_title(f'Global agreement approaches 1 while top-{k} RIDI remains 1')
ax.grid(alpha=.2)
ax.legend()
plt.show()

## RIDI itself is simple

RIDI is Jaccard distance between two finite decision sets. Zero means identical selected identities; one means disjoint selections. It complements performance and robustness metrics rather than replacing them.

In [ ]:
def RIDI(left, right):
    left, right = set(left), set(right)
    return 0.0 if not left and not right else 1 - len(left & right) / len(left | right)

print('same sets:    ', RIDI(['a', 'b'], ['a', 'b']))
print('one replaced: ', RIDI(['a', 'b'], ['a', 'c']))
print('disjoint:     ', RIDI(['a', 'b'], ['c', 'd']))

## Next step

The full [`ridi-audit`](https://github.com/adeebnoor/ridi) toolkit adds deterministic ties, changed-slot reporting, the sufficient score-margin certificate, and the exact identity–utility frontier. RIDI measures turnover; it does not by itself establish correctness, fairness, clinical utility, or harm.